In [5]:
REPO_ROOT = Path.cwd().parent if 'data' in str(Path.cwd()) else Path.cwd()
SRC_DIR = REPO_ROOT / 'src'
DATA_DIR = Path(REPO_ROOT,'data/wellness_centre')
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [9]:
from orchestration import MigrationOrchestrator
orchestrator = MigrationOrchestrator(DATA_DIR)

In [11]:
# Test batch
test_results = orchestrator.process_batch(limit=10)
print(orchestrator.generate_report(test_results))

Starting batch processing (limit: 10)...

1. Loading CSV data...
   Loaded 10 events
   Loaded 10 room bookings
   Loaded 10 egg sales

2. Generating invoices...
   Generated 10 event invoices
   Generated 10 room invoices
   Generated 10 egg invoices
   Total: 30 invoices

3. Calculating totals...
   Event revenue: KES 597,400.00
   Room revenue: KES 156,600.00
   Egg revenue: KES 14,950.00
   Total revenue: KES 768,950.00
WELLNESS CENTRE MIGRATION REPORT

DATA LOADED:
  Events:          10
  Room Bookings:   10
  Egg Sales:       10

INVOICES GENERATED:
  Event Invoices:   10
  Room Invoices:    10
  Egg Invoices:     10
  Total:            30

REVENUE BREAKDOWN:
  Events:
    Subtotal: KES 515,000.00
    Tax:      KES 82,400.00
    Total:    KES 597,400.00

  Rooms:
    Subtotal: KES 135,000.00
    Tax:      KES 21,600.00
    Total:    KES 156,600.00

  Eggs:
    Subtotal: KES 14,950.00
    Tax:      KES 0.00
    Total:    KES 14,950.00

COMBINED TOTALS:
  Subtotal:    KES 664,950.0

In [12]:
# Full migration - all 182 records
full_results = orchestrator.process_batch()
print(orchestrator.generate_report(full_results))

Starting batch processing (limit: ALL)...

1. Loading CSV data...
   Loaded 25 events
   Loaded 54 room bookings
   Loaded 103 egg sales

2. Generating invoices...
   Generated 25 event invoices
   Generated 54 room invoices
   Generated 103 egg invoices
   Total: 182 invoices

3. Calculating totals...
   Event revenue: KES 1,969,680.00
   Room revenue: KES 718,040.00
   Egg revenue: KES 136,840.00
   Total revenue: KES 2,824,560.00
WELLNESS CENTRE MIGRATION REPORT

DATA LOADED:
  Events:          25
  Room Bookings:   54
  Egg Sales:      103

INVOICES GENERATED:
  Event Invoices:   25
  Room Invoices:    54
  Egg Invoices:    103
  Total:           182

REVENUE BREAKDOWN:
  Events:
    Subtotal: KES 1,698,000.00
    Tax:      KES 271,680.00
    Total:    KES 1,969,680.00

  Rooms:
    Subtotal: KES 619,000.00
    Tax:      KES 99,040.00
    Total:    KES 718,040.00

  Eggs:
    Subtotal: KES 136,840.00
    Tax:      KES 0.00
    Total:    KES 136,840.00

COMBINED TOTALS:
  Subtotal: 

In [13]:
!pip install frappeclient --break-system-packages

In [17]:
# Modify Cell 5 - print full error
def submit_to_erpnext(orchestrator, results, api_key, api_secret):
    """Submit generated invoices to ERPNext."""
    from frappeclient import FrappeClient
    
    url = "http://erpnext-frontend:8080"
    domain = "well.rosslyn.cloud"
    
    client = FrappeClient(url)
    client.authenticate(api_key, api_secret)
    client.session.headers.update({"Host": domain})
    
    all_invoices = (
        results['invoices']['event_invoices'] +
        results['invoices']['room_invoices'] +
        results['invoices']['egg_invoices']
    )
    
    # Test with just ONE invoice to see full error
    inv = all_invoices[0]
    payload = inv.to_erpnext_format()
    
    print("Testing first invoice...")
    print(f"Payload keys: {payload.keys()}")
    
    try:
        doc = client.insert(payload)
        print(f"✓ Success: {doc['name']}")
    except Exception as e:
        print(f"Full error:\n{e}")
        import traceback
        traceback.print_exc()

In [19]:
# Your API credentials
API_KEY = "c7edf88efe48324"
API_SECRET = "31fb771ebeec0d6"

# Submit
submission_results = submit_to_erpnext(orchestrator, full_results, API_KEY, API_SECRET)

Testing first invoice...
Payload keys: dict_keys(['doctype', 'customer', 'customer_name', 'posting_date', 'due_date', 'items', 'taxes', 'remarks'])
Full error:
["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/api/__init__.py\", line 52, in handle\n    data = endpoint(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/api/v1.py\", line 46, in create_doc\n    return frappe.new_doc(doctype, **data).insert()\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/model/document.py\", line 309, in insert\n    self.run_before_save_methods()\n  File \"apps/frappe/frappe/model/document.py\", line 1140, in run_before_save_methods\n    self.run_method(\"validate\")\n  File \"apps/frappe/frappe/model/document.py\", line 1011, in run_method\n    out = Document.hook(fn)(self, *args, *

Traceback (most recent call last):
  File "/tmp/ipykernel_1253717/3851143043.py", line 27, in submit_to_erpnext
    doc = client.insert(payload)
          ^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/venvs/erpnext/lib/python3.11/site-packages/frappeclient/frappeclient.py", line 99, in insert
    return self.post_process(res)
           ^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/venvs/erpnext/lib/python3.11/site-packages/frappeclient/frappeclient.py", line 285, in post_process
    raise FrappeException(rjson['exc'])
frappeclient.frappeclient.FrappeException: ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/api/__init__.py\", line 52, in handle\n    data = endpoint(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/api/v1.py\", line 46, in create_doc\n    return frappe.new_doc(doctype, **data).inser

In [16]:
# Modify Cell 5 - print full error
def submit_to_erpnext(orchestrator, results, api_key, api_secret):
    """Submit generated invoices to ERPNext."""
    from frappeclient import FrappeClient
    
    url = "http://erpnext-frontend:8080"
    domain = "well.rosslyn.cloud"
    
    client = FrappeClient(url)
    client.authenticate(api_key, api_secret)
    client.session.headers.update({"Host": domain})
    
    all_invoices = (
        results['invoices']['event_invoices'] +
        results['invoices']['room_invoices'] +
        results['invoices']['egg_invoices']
    )
    
    # Test with just ONE invoice to see full error
    inv = all_invoices[0]
    payload = inv.to_erpnext_format()
    
    print("Testing first invoice...")
    print(f"Payload keys: {payload.keys()}")
    
    try:
        doc = client.insert(payload)
        print(f"✓ Success: {doc['name']}")
    except Exception as e:
        print(f"Full error:\n{e}")
        import traceback
        traceback.print_exc()